In [1]:
# =============================================================================
# CELDA 1 — Configuración de rutas para importar el código del proyecto
# =============================================================================

from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / "backend" / "app").is_dir():
    BACKEND_ROOT = REPO_ROOT / "backend"
else:
    REPO_ROOT = REPO_ROOT.parent
    BACKEND_ROOT = REPO_ROOT / "backend"

sys.path.insert(0, str(BACKEND_ROOT))

In [2]:
# =============================================================================
# CELDA 2 — Importaciones
# =============================================================================

from time import perf_counter

from app.simulation.core.data_processing import (
    eliminar_valores_fuera_de_indices,
    get_processing_result_from_csv_dir,
    normalize_mode_of_operation_in_csv_dir,
    reorder_activity_ratio_csvs_for_dataportal,
    run_data_processing_from_excel,
    strip_whitespace_in_set_csvs,
)
from app.simulation.core.instance_builder import build_instance
from app.simulation.core.model_definition import create_abstract_model
from app.simulation.core.solver import (
    _apply_highspy_solution_to_instance,
    _solve_with_appsi_highs,
    _solve_with_direct_highspy,
    write_lp_file,
)
from app.simulation.core.solver_config import resolve_highs_config
from app.core.config import get_settings
from pyomo.environ import Var
import pyomo.environ as pyo
from dataclasses import replace

import shutil
import tempfile
import zipfile

import highspy

from pathlib import Path

In [3]:
# =============================================================================
# CELDA 3 — Funciones auxiliares
# =============================================================================


def preprocess_csv_dir(csv_dir: str | Path) -> None:
    """Limpia CSVs ya extraídos (caso regional desde ZIP)."""
    csv_dir = str(csv_dir)
    reorder_activity_ratio_csvs_for_dataportal(csv_dir)
    normalize_mode_of_operation_in_csv_dir(csv_dir)
    strip_whitespace_in_set_csvs(csv_dir)
    eliminar_valores_fuera_de_indices(csv_dir)


def csv_dir_from_zip(csv_zip: Path, work_dir: Path) -> Path:
    """Descomprime CSV.zip → carpeta work_dir/csv con *.csv."""
    csv_dir = work_dir / "csv"
    extract_to = work_dir / "_zip_extract"
    extract_to.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(csv_zip, "r") as zf:
        zf.extractall(extract_to)
    nested = extract_to / "CSV"
    if not nested.is_dir():
        raise FileNotFoundError(f"No se encontró carpeta CSV/ dentro de {csv_zip}")
    csv_dir.mkdir(parents=True, exist_ok=True)
    for item in nested.iterdir():
        shutil.move(str(item), str(csv_dir / item.name))
    shutil.rmtree(extract_to)
    return csv_dir


def csv_dir_from_excel(excel_path: Path, work_dir: Path):
    """Excel SAND → CSVs + ProcessingResult (pipeline nacional)."""
    csv_dir = work_dir / "csv"
    csv_dir.mkdir(parents=True, exist_ok=True)
    proc = run_data_processing_from_excel(excel_path, str(csv_dir))
    return csv_dir, proc

In [4]:
# =============================================================================
# CELDA 4 — Cargar datos y construir la instancia Pyomo
# =============================================================================

CSV_ZIP_REGIONAL = REPO_ROOT / "CSV.zip"
EXCEL_NACIONAL = REPO_ROOT / "01-04-2026 SAND BASE v10.xlsx"


def _require_input_file(path: Path, label: str) -> Path:
    if not path.is_file():
        raise FileNotFoundError(f"{label} no encontrado: {path}")
    return path


# --- Fuente: comenta UNA línea y deja activa la otra ---
MODO = "regional"
#MODO = "nacional"

WORK_DIR = Path(tempfile.mkdtemp(prefix="osemosys_pruebas_"))
print("MODO:", MODO, "| WORK_DIR:", WORK_DIR)

if MODO == "regional":
    _require_input_file(CSV_ZIP_REGIONAL, "CSV regional (ZIP)")
    CSV_DIR = csv_dir_from_zip(CSV_ZIP_REGIONAL, WORK_DIR)
    preprocess_csv_dir(CSV_DIR)
    proc = get_processing_result_from_csv_dir(str(CSV_DIR))
elif MODO == "nacional":
    _require_input_file(EXCEL_NACIONAL, "Excel SAND nacional")
    CSV_DIR, proc = csv_dir_from_excel(EXCEL_NACIONAL, WORK_DIR)
else:
    raise ValueError(f'MODO debe ser "regional" o "nacional", recibí: {MODO!r}')

print("CSV_DIR:", CSV_DIR, "| archivos .csv:", len(list(CSV_DIR.glob("*.csv"))))
print(f"has_storage={proc.has_storage}, has_udc={proc.has_udc}")

t0 = perf_counter()
model = create_abstract_model(
    has_storage=proc.has_storage,
    has_udc=proc.has_udc,
)
instance = build_instance(
    model,
    str(CSV_DIR),
    has_storage=proc.has_storage,
    has_udc=proc.has_udc,
)
print(f"Instancia en {perf_counter() - t0:.1f} s")
print("Años:", sorted(instance.YEAR.data()))
print("Regiones:", list(instance.REGION.data()))

MODO: regional | WORK_DIR: C:\Users\SGISAS~1\AppData\Local\Temp\osemosys_pruebas_ygnunn65
CSV_DIR: C:\Users\SGISAS~1\AppData\Local\Temp\osemosys_pruebas_ygnunn65\csv | archivos .csv: 57
has_storage=True, has_udc=False
           0 seconds to construct Set YEAR; 1 index total
           0 seconds to construct Set TECHNOLOGY; 1 index total
           0 seconds to construct Set TIMESLICE; 1 index total
           0 seconds to construct Set FUEL; 1 index total
           0 seconds to construct Set EMISSION; 1 index total
           0 seconds to construct Set MODE_OF_OPERATION; 1 index total
           0 seconds to construct Set REGION; 1 index total
           0 seconds to construct Set STORAGE; 1 index total
           0 seconds to construct Set SEASON; 1 index total
           0 seconds to construct Set DAYTYPE; 1 index total
           0 seconds to construct Set DAILYTIMEBRACKET; 1 index total
           0 seconds to construct Set STORAGEINTRADAY; 1 index total
           0 seconds to c

In [5]:
# =============================================================================
# CELDA 5 — Exportar instancia → archivo .lp
# =============================================================================
# Requiere celda 4 ejecutada (variables: instance, WORK_DIR).

from app.simulation.core.solver import write_lp_file

LP_PATH = WORK_DIR / "model.lp"

t0 = perf_counter()
lp_written = write_lp_file(instance, LP_PATH)

print("Archivo LP:", lp_written)
print(f"Tamaño: {lp_written.stat().st_size / (1024 * 1024):.2f} MB")
print(f"Tiempo escritura: {perf_counter() - t0:.1f} s")

Archivo LP: C:\Users\SGISAS~1\AppData\Local\Temp\osemosys_pruebas_ygnunn65\model.lp
Tamaño: 1055.61 MB
Tiempo escritura: 137.7 s


In [6]:
h = highspy.Highs()

In [7]:
t0 = perf_counter()
h.readModel(str(LP_PATH))
print(f"Tiempo lectura LP: {perf_counter() - t0:.1f} s")

Tiempo lectura LP: 57.2 s


In [ ]:
h.clearSolver()
h.resetOptions()

# Activar opciones de log
h.setOptionValue('output_flag', True)
h.setOptionValue('log_to_console', True)
h.setOptionValue('log_file', 'log_highs.txt')

# 1. Ajuste de Hilos (Threads) - PAMI se acelera con esto.
# Pon aquí el máximo de tu procesador (ej: 16 si tienes un octacore con multithreading)
#h.setOptionValue('threads', 16)

# 2. Relajar tolerancias. 
# En OSeMOSYS, pelear por el decimal 1e-7 en un objetivo de 2 millones es 
# un desperdicio de iteraciones. Bajarlo a 1e-5 cerrará el modelo más rápido.
h.setOptionValue('primal_feasibility_tolerance', 1e-5)
h.setOptionValue('dual_feasibility_tolerance', 1e-5)

# 3. Forzar el uso de Simplex Dual con PAMI explícitamente y desactivar el Crossover
#h.setOptionValue('solver', 'simplex')
#h.setOptionValue('simplex_strategy', 1) # 1 es Dual simplex
#h.setOptionValue('run_crossover', 'off')

t0 = perf_counter()
h.run()
print(f"Tiempo resolución: {perf_counter() - t0:.1f} s")


In [12]:
print("Estado:", h.getModelStatus())
print("Objetivo:", h.getObjectiveValue())
print("Número de nucleos:", h.getOptionValue('threads'))

Estado: HighsModelStatus.kOptimal
Objetivo: 1722773.7086489273
Número de nucleos: (<HighsStatus.kOk: 0>, 0)


In [10]:
n_vars = sum(1 for _ in instance.component_data_objects(Var, active=True))
print(f"Variables activas en Pyomo: {n_vars:,}")

t0 = perf_counter()
obj_pyomo, dual_map = _apply_highspy_solution_to_instance(instance, h)
t_map = perf_counter() - t0

print(f"Mapeo solución → Pyomo: {t_map:.1f} s")
print(f"Objetivo tras mapeo: {obj_pyomo:,.4f}")
print(f"Duales cargados: {len(dual_map):,}")

Variables activas en Pyomo: 3,189,168
Mapeo solución → Pyomo: 31.6 s
Objetivo tras mapeo: 1,722,773.7086
Duales cargados: 3,390,685
